# Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019C29046CF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019C29047770>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="the title of movie")
    year:int=Field(description="this year movie was released")
    director:str=Field(description="director of the movie")
    rating:float=Field(description="the movies rating out of ten")



In [5]:
model_structure =model.with_structured_output(Movie)
model_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019C29046CF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019C29047770>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the tit

In [6]:
model_structure.invoke("provide details about  movvie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [7]:
# similarly if we use nly model.invoke we will get 
model.invoke("provide detailabout movie inception")

AIMessage(content="**Inception**  \n*Release Date:* July\u202f16\u202f2010 (USA)  \n*Director & Writer:* Christopher\u202fNolan  \n*Genre:* Science‑fiction thriller / Psychological action  \n*Runtime:* 2\u202fh\u202f28\u202fmin  \n*Budget:* ~$160\u202fmillion  \n*Box‑Office:* ~$828\u202fmillion worldwide  \n\n---\n\n## 1.  Plot Overview\n\nThe film follows **Dom\u202fCobb** (Leonardo\u202fDiCaprio), a master “extractor” who enters the subconscious of others to steal or plant ideas. After a highly successful career, Cobb is offered a chance to return home to his children in exchange for a seemingly impossible task: **inception**—the planting of an idea deep enough that the target will believe it is their own.\n\n### Core Story Arc\n\n| Act | Key Events | Dream Layer |\n|-----|------------|-------------|\n| 1️⃣ | Cobb is approached by Saito (Ken Watanabe) to perform inception on **Robert Fischer** (Cillian\u202fBally), the heir to a business empire. | 1st (City) |\n| 2️⃣ | Cobb assembles

# Message output alongside parsed structure 

In [12]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="the title of movie")
    year:int=Field(...,description="the year movie was released")
    directer:str=Field(...,description="the directer of movie")
    rating:float=Field(...,description="the rating of movie out of ten")
model_structure = model.with_structured_output(Movie,include_raw=True)
responce = model_structure.invoke("provide details about movie inception")
responce

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to use the function. Provide details about movie Inception. We'll call functions.Movie with parameters: directer: Christopher Nolan, rating: maybe 8.8, title: Inception, year: 2010.", 'tool_calls': [{'id': 'fc_2bf9b8ba-576e-4338-a3da-279d615aef92', 'function': {'arguments': '{"directer":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 160, 'total_tokens': 247, 'completion_time': 0.095251836, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.007799321, 'prompt_tokens_details': None, 'queue_time': 0.376333945, 'total_time': 0.103051157}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057b5-e5ca-79d3-bb92-9b38

# Nested structure

In [15]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str
class Movies(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float|None =Field(None,description="budget is millio usd")
model_structure =model.with_structured_output(Movies)
respoce =model_structure.invoke("provide details about movie inception")
responce

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to use the function. Provide details about movie Inception. We'll call functions.Movie with parameters: directer: Christopher Nolan, rating: maybe 8.8, title: Inception, year: 2010.", 'tool_calls': [{'id': 'fc_2bf9b8ba-576e-4338-a3da-279d615aef92', 'function': {'arguments': '{"directer":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 160, 'total_tokens': 247, 'completion_time': 0.095251836, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.007799321, 'prompt_tokens_details': None, 'queue_time': 0.376333945, 'total_time': 0.103051157}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057b5-e5ca-79d3-bb92-9b38

# TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [16]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str,..., "the tittle of movie"]
    year:Annotated[int,...,"the year in which movie was release"]
    director:Annotated[str,...,"the director of the movie"]
    rating:Annotated[float,...,"the movie'srating out of ten"]

model_structure = model.with_structured_output(MovieDict)
model_structure.invoke("provide details of movie avenger ")

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

# DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the `@dataclass` decorator.

In [1]:
import os
os.environ["GOOGLE_API_KEY"]= os.getenv("GOOGLE_API_KEY")


In [3]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent
class contactInfo(BaseModel):
    """contact info of a person"""
    name:str=Field(description="the name of person")
    email:str=Field(description="the email adress of a person")
    phone:str=Field(description="the phone number pf person")

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=contactInfo
)
result =agent.invoke({
    "messages":[{"role":"user","content":"Extract info from :John doe , john@example.com , (555) 123-4567"}]
})
result

{'messages': [HumanMessage(content='Extract info from :John doe , john@example.com , (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6b5c7018-3f3b-4e70-99f2-35b8604f6767'),
  AIMessage(content='{"name":"John doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05848-22f8-7ed0-85cd-86eedeacd182-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 208, 'total_tokens': 236, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 177}})],
 'structured_response': contactInfo(name='John doe', email='john@example.com', phone='(555) 123-4567')}

In [7]:
# Type Dict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class contactInfo(TypedDict):
    name:str
    email:str
    phone:str
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=contactInfo
)
result = agent.invoke(
    { "messages":[{"role":"user","content":"Extract info from :John doe , john@example.com , (555) 123-4567"}]}
)
result

{'messages': [HumanMessage(content='Extract info from :John doe , john@example.com , (555) 123-4567', additional_kwargs={}, response_metadata={}, id='f012e827-fc98-4e95-8b2c-77f283d7b98f'),
  AIMessage(content='{"name":"John doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05851-274a-7d31-b2be-bc487480c80b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 175, 'total_tokens': 203, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 144}})],
 'structured_response': {'name': 'John doe',
  'email': 'john@example.com',
  'phone': '(555) 123-4567'}}

In [9]:
##now with  data class 
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class contactInfo:
    name:str
    email:str
    phone:str

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=contactInfo
)

result = agent.invoke({
     "messages":[{"role":"user","content":"Extract info from :John doe , john@example.com , (555) 123-4567"}]
})

result   

{'messages': [HumanMessage(content='Extract info from :John doe , john@example.com , (555) 123-4567', additional_kwargs={}, response_metadata={}, id='65afd5ec-0211-4ad0-9252-0974f4abaa2a'),
  AIMessage(content='{"name":"John doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05857-ba10-7891-9790-a75b5e196170-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 172, 'total_tokens': 200, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 141}})],
 'structured_response': contactInfo(name='John doe', email='john@example.com', phone='(555) 123-4567')}